# Testing and Evaluating the model B : TB Detector (densnet - 121)

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from torch.amp.autocast_mode import autocast


print("All libraries imported successfully.") 

All libraries imported successfully.


In [2]:
# Config
BATCH_SIZE = 128
NUM_WORKERS = 0
USE_AMP = True


In [3]:
# DATASET
class TBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((224, 224), dtype=np.uint8)
        img = np.stack([img, img, img], axis=2)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.long)


val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [4]:
# model
def create_model_b():
    model = models.densenet121(weights=None)
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential( # type: ignore
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 3)
    )
    return model

In [9]:
# Main
def main():
    
    BASE_DIR = os.path.abspath("../data_pr/preprocessed")
    MASTER_CSV = os.path.join(BASE_DIR, 'model_b_dataset.csv')
    MODEL_PATH = os.path.abspath("../models/best_model_b.pth")
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {DEVICE}")
    
    # Load test split
    df = pd.read_csv(MASTER_CSV)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    print(f"Test samples: {len(test_df)}")
    
    # Dataset & Loader
    test_dataset = TBDataset(test_df, transform=val_transform)
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,   # 0
        pin_memory=True
    )
    
    # Model
    model = create_model_b().to(DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"✅ Loaded model from epoch {checkpoint['epoch'] + 1}")
    
    # Evaluate
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP):
                outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    all_probs = np.array(all_probs) 
    all_probs = all_probs/all_probs.sum(axis=1, keepdims=True)  # Normalize probabilities
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    
    print(f"\nTest Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  AUC-ROC:  {auc:.4f}")
    
    class_names = ['Normal', 'TB', 'Abnormal']
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(pd.DataFrame(cm, index=class_names, columns=class_names).to_string())

In [10]:
if __name__ == "__main__":
    main()

Device: cuda
Test samples: 11391
✅ Loaded model from epoch 8

Test Results:
  Accuracy: 0.9344
  F1-Score: 0.9241
  AUC-ROC:  0.9844

Classification Report:
              precision    recall  f1-score   support

      Normal       0.92      0.99      0.95      5629
          TB       0.94      0.92      0.93      3412
    Abnormal       0.98      0.81      0.89      2350

    accuracy                           0.93     11391
   macro avg       0.95      0.91      0.92     11391
weighted avg       0.94      0.93      0.93     11391


Confusion Matrix:
          Normal    TB  Abnormal
Normal      5592    13        24
TB           257  3148         7
Abnormal     248   198      1904
